# 0 - Immune Infiltration Landscape

**Purpose**: CIDER Score distribution across cancer types in the full Hartwig database (~8,000 patients). Used as Figure 2A technical validation — independent of the CPI analysis cohort.

| | |
|---|---|
| **Inputs** | `CIDER_PROJ_DIR/ready.rds` |
| **Outputs** | `figs/landscape.png` |

In [2]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))
source(paste0(dirname(getwd()), "/helpers/themes.r"))

# Ensure figures output directory exists
dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)

In [3]:
library(ggpubr)
library(patchwork)

# 0 - Read data

In [ ]:
# Read full Hartwig database — all patients, not just CPI.
# cohort_pan / cohort_pan_go are the pan-database cohort labels.
ready <- readRDS(paste0(CIDER_PROJ_DIR, "ready.rds")) |>
  mutate(gp = factor(gp, levels = GP_LEVELS))

# 1 - Immune infiltration across cohorts

In [ ]:
# Order cohorts by median CIDER T-cell Score descending.
# Pan-Cancer (all patients) is pinned to the leftmost position.
cohort_counts <- ready |>
  filter(cohort_pan %in% PAN_LEVELS, gp == "T-cell") |>
  group_by(cohort_pan) |>
  summarise(
    n   = n_distinct(sampleId),
    med = median(cider_score_depth_adj, na.rm = TRUE),
    .groups = "drop"
  ) |>
  mutate(label = paste0(cohort_pan, " (N = ", format(n, big.mark = ",", trim = TRUE), ")"))

# Pan-Cancer row: all patients pooled
pan_n     <- n_distinct(ready$sampleId[ready$gp == "T-cell"])
pan_label <- paste0("Pan-Cancer (N = ", format(pan_n, big.mark = ",", trim = TRUE), ")")

# Other pinned to far right; remaining cohorts ordered by median descending
other_label <- cohort_counts |>
  filter(cohort_pan == "Other") |>
  pull(label)

tumour_order <- cohort_counts |>
  filter(cohort_pan != "Other") |>
  arrange(desc(med)) |>
  pull(label)

plot_levels <- c(pan_label, tumour_order, other_label)

# Add Pan-Cancer rows by duplicating all T-cell rows
pan_rows <- ready |>
  filter(cohort_pan %in% PAN_LEVELS, gp == "T-cell") |>
  mutate(label = pan_label)

ready_plot <- ready |>
  filter(cohort_pan %in% PAN_LEVELS, gp == "T-cell") |>
  left_join(select(cohort_counts, cohort_pan, label), by = "cohort_pan") |>
  bind_rows(pan_rows) |>
  mutate(plot_label = factor(label, levels = plot_levels))

# Cap y-axis at 99.9th percentile to prevent two extreme outliers from
# compressing the display of the remaining ~8,000 points. coord_cartesian
# clips the view only — all data are still used by geoms and stats.
y_cap <- quantile(ready_plot$cider_score_depth_adj, 0.999, na.rm = TRUE)

# Compute per-cohort medians for all cell type groups
median_labels <- ready_plot |>
  group_by(plot_label, gp) |>
  summarise(med = median(cider_score_depth_adj, na.rm = TRUE), .groups = "drop") |>
  mutate(med_label = as.character(round(med)))

# Jitter data: cap at 150 per cohort to avoid overplotting
jitter_data_landscape <- ready_plot |>
  dplyr::group_by(plot_label) |>
  dplyr::slice_sample(n = 150) |>
  dplyr::ungroup()

p_landscape <- ready_plot |>
  ggplot(aes(x = plot_label, y = cider_score_depth_adj)) +
  # Points bumped up (0.12 -> 0.2 alpha) and the box lightened (0.7 -> 0.5
  # alpha) so the underlying jitter is visible through the boxplot, not
  # just around its edges.
  geom_jitter(
    data   = jitter_data_landscape,
    width  = 0.18, alpha = 0.2, size = 0.35, colour = COL_NEGATIVE
  ) +
  geom_boxplot(fill = COL_NEGATIVE, colour = COL_INK, outlier.shape = NA, alpha = 0.5) +
  # Median labels above each boxplot — dodge width matches geom_boxplot default
  geom_text(
    data     = median_labels,
    aes(x = plot_label, y = med, label = med_label, group = gp),
    inherit.aes = FALSE,
    position = position_dodge(width = 0.75),
    vjust    = -0.5, size = 2.2, fontface = "bold", colour = COL_INK
  ) +
  scale_y_continuous(trans = log10p1_trans) +
  coord_cartesian(ylim = c(0, y_cap)) +
  go_theme +
  theme(axis.text.x = element_text(angle = 0, hjust = 0.5),
        plot.subtitle = element_text(hjust = 0.5)) +
  scale_x_discrete(guide = guide_axis(n.dodge = 3)) +
  labs(
    title    = "CIDER T-cell Score Across Cancer Types - Hartwig Medical Foundation",
    subtitle = paste0("All Hartwig patients with WGS; n = ", format(pan_n, big.mark = ",", trim = TRUE)),
    x = "Cohort", y = "CIDER T-cell Score"
  )

set_plot_dims(14, 5); p_landscape
saveRDS(p_landscape, paste0(CIDER_PROJ_DIR, "figs/landscape.rds"))

In [ ]:
# 2 - Landscape with T-cell loci breakdown
# Shows overall T-cell + per-locus scores (TRA/TRD, TRB, TRG) as facet rows.
# Cohort order matches the main landscape (sorted by overall T-cell median).

TCELL_LOCI_LEVELS <- c("T-cell", "T-cell (TRA/TRD)", "T-cell (TRB)", "T-cell (TRG)")
TCELL_LOCI_LABELS <- c("T-cell (all loci)", "T-cell (TRA/TRD)", "T-cell (TRB)", "T-cell (TRG)")

pan_rows_loci <- ready |>
  dplyr::filter(cohort_pan %in% PAN_LEVELS, gp %in% TCELL_LOCI_LEVELS) |>
  dplyr::mutate(label = pan_label)

ready_plot_loci <- ready |>
  dplyr::filter(cohort_pan %in% PAN_LEVELS, gp %in% TCELL_LOCI_LEVELS) |>
  dplyr::left_join(dplyr::select(cohort_counts, cohort_pan, label), by = "cohort_pan") |>
  dplyr::bind_rows(pan_rows_loci) |>
  dplyr::mutate(
    plot_label = factor(label, levels = plot_levels),
    gp         = factor(gp, levels = TCELL_LOCI_LEVELS,
                         labels = TCELL_LOCI_LABELS)
  )

y_cap_loci <- quantile(ready_plot_loci$cider_score_depth_adj, 0.999, na.rm = TRUE)

median_labels_loci <- ready_plot_loci |>
  dplyr::group_by(plot_label, gp) |>
  dplyr::summarise(med = median(cider_score_depth_adj, na.rm = TRUE), .groups = "drop") |>
  dplyr::mutate(med_label = as.character(round(med)))

# Jitter data: cap at 150 per cohort × locus group
jitter_data_loci <- ready_plot_loci |>
  dplyr::group_by(plot_label, gp) |>
  dplyr::slice_sample(n = 150) |>
  dplyr::ungroup()

p_landscape_loci <- ready_plot_loci |>
  ggplot2::ggplot(ggplot2::aes(x = plot_label, y = cider_score_depth_adj)) +
  ggplot2::geom_jitter(
    data   = jitter_data_loci,
    width  = 0.18, alpha = 0.12, size = 0.3, colour = COL_NEGATIVE
  ) +
  ggplot2::geom_boxplot(fill = COL_NEGATIVE, colour = COL_INK,
                        outlier.shape = NA, alpha = 0.7) +
  ggplot2::geom_text(
    data        = median_labels_loci,
    ggplot2::aes(x = plot_label, y = med, label = med_label),
    inherit.aes = FALSE,
    vjust = -0.5, size = 1.8, fontface = "bold", colour = COL_INK
  ) +
  ggplot2::facet_grid(rows = ggplot2::vars(gp), scales = "free_y") +
  ggplot2::scale_y_continuous(trans = log10p1_trans) +
  ggplot2::coord_cartesian(ylim = c(0, y_cap_loci)) +
  go_theme +
  ggplot2::theme(
    axis.text.x    = ggplot2::element_text(angle = 0, hjust = 0.5),
    plot.subtitle  = ggplot2::element_text(hjust = 0.5),
    strip.text.y   = ggplot2::element_text(size = 7, face = "bold")
  ) +
  ggplot2::scale_x_discrete(guide = ggplot2::guide_axis(n.dodge = 3)) +
  ggplot2::labs(
    title    = "CIDER T-cell Score Across Cancer Types — Overall and Per Locus",
    subtitle = paste0("All Hartwig patients with WGS; n = ",
                      format(pan_n, big.mark = ",", trim = TRUE)),
    x = "Cohort", y = "CIDER Score"
  )

set_plot_dims(14, 9); p_landscape_loci
saveRDS(p_landscape_loci, paste0(CIDER_PROJ_DIR, "figs/landscape_loci.rds"))


In [ ]:
# 3 - Landscape: all T-cell loci side-by-side per cohort (no faceting)
# Boxes are dodged within each cohort, filled by locus.
# Same y-axis scale and cohort order as the main landscape (p_landscape).

# 4-level palette for T-cell locus groups: with no natural positive/negative
# ordering among loci, cycle the 4 non-reserved colours in a fixed order
# (POSITIVE, NEGATIVE, MID, ACCENT) rather than inventing a 5th hue.
loci_colors <- c(
  "T-cell (all loci)" = COL_POSITIVE,
  "T-cell (TRA/TRD)"  = COL_NEGATIVE,
  "T-cell (TRB)"      = COL_MID,
  "T-cell (TRG)"      = COL_ACCENT
)

# Jitter: cap at 75 per cohort × locus to avoid overplotting with 4 groups
jitter_loci_dodged <- ready_plot_loci |>
  dplyr::group_by(plot_label, gp) |>
  dplyr::slice_sample(n = 75) |>
  dplyr::ungroup()

p_landscape_loci_dodged <- ready_plot_loci |>
  ggplot2::ggplot(ggplot2::aes(x = plot_label, y = cider_score_depth_adj, fill = gp)) +
  ggplot2::geom_jitter(
    data     = jitter_loci_dodged,
    ggplot2::aes(colour = gp),
    position = ggplot2::position_jitterdodge(jitter.width = 0.1, dodge.width = 0.8),
    alpha = 0.15, size = 0.3, show.legend = FALSE
  ) +
  ggplot2::scale_colour_manual(values = loci_colors, guide = "none") +
  ggplot2::geom_boxplot(
    outlier.shape = NA,
    alpha         = 0.7,
    position      = ggplot2::position_dodge(width = 0.8)
  ) +
  ggplot2::scale_fill_manual(values = loci_colors, name = "T-cell Loci") +
  ggplot2::scale_y_continuous(trans = log10p1_trans) +
  ggplot2::coord_cartesian(ylim = c(0, y_cap_loci)) +
  go_theme +
  ggplot2::theme(
    axis.text.x   = ggplot2::element_text(angle = 0, hjust = 0.5),
    plot.subtitle = ggplot2::element_text(hjust = 0.5)
  ) +
  ggplot2::scale_x_discrete(guide = ggplot2::guide_axis(n.dodge = 3)) +
  ggplot2::labs(
    title    = "CIDER T-cell Score Across Cancer Types \u2014 T-cell Loci",
    subtitle = paste0("All Hartwig patients with WGS; n = ",
                      format(pan_n, big.mark = ",", trim = TRUE)),
    x = "Cohort", y = "CIDER T-cell Score"
  )

set_plot_dims(14, 5); p_landscape_loci_dodged
saveRDS(p_landscape_loci_dodged,
        paste0(CIDER_PROJ_DIR, "figs/landscape_loci_dodged.rds"))
